![JohnSnowLabs](https://nlp.johnsnowlabs.com/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp-workshop/blob/master/tutorials/vlm-workshop/8.kyc_id_verification.ipynb)

# 🪪 KYC Identity Verification

> Financial institutions spend `$500M+/yr` on KYC compliance. Manual ID verification is 2-3 minutes per document, error-prone, and doesn't scale across languages or countries.

**What you'll see:**
- Any ID document (passport, national ID) → 21 structured fields in JSON in under 3 seconds
- 10 countries, 10 scripts — Albanian, Azerbaijani, Estonian, Finnish, Greek, Latvian, Russian, Serbian, Slovak, Spanish
- Automatic English transliteration of non-Latin fields (20/20 documents)
- Honest benchmarks: where it excels (expiry dates: 87.5%, birth dates: 75%) and where it struggles (authority: 33.3%)

```mermaid
flowchart LR
    A["🖼️ ID Document Image<br>(any country)"] --> B["jsl_vision_ocr_1.0<br>(21-field extraction)"]
    B --> C["Transliteration<br>(non-Latin → English)"]
    C --> D["📊 Accuracy<br>vs MIDV-2020 GT"]
    style B fill:#6366f1,color:#fff,stroke:#4f46e5
    style C fill:#f59e0b,color:#fff,stroke:#d97706
    style D fill:#10b981,color:#fff,stroke:#059669
```

| Segment | Dataset | Docs | What Happens |
|---------|---------|:----:|--------------|
| ID field extraction | [MIDV-2020](https://arxiv.org/abs/2107.00396) | 20 | Extract 21 fields from 10-country multilingual ID documents |

**Models**: JSL Vision OCR 1.0 (field extraction) · also benchmarked: Claude Opus 4.5, Claude Sonnet 4.6, GPT-5.4
**Evaluation**: Per-field accuracy vs ground truth, translation coverage

**Bottom line**: At 10K IDs/month, saves ~`$172K/yr` over manual entry. At 100K/month (large bank): ~`$1.7M/yr`.


---
## 📦 Setup & Configuration

In [ ]:
# -- Colab Setup ---------------------------------------------------------------
# Run this cell once to install dependencies and download data.
# Skipped automatically when running locally.
import os, sys
if "COLAB_RELEASE_TAG" in os.environ:
    # 1) Python dependencies
    !pip install -q pandas numpy tqdm pillow datasets matplotlib seaborn scikit-learn openai python-dotenv faiss-cpu requests PyMuPDF

    # 2) Source files (utils/, config)
    !wget -qO git_downloads.tar.gz "https://ckl-emr-bucket.s3.us-east-1.amazonaws.com/vlm-workshop/git_downloads.tar.gz"
    !tar xzf git_downloads.tar.gz && rm git_downloads.tar.gz

    # 3) Cached predictions + datasets
    !wget -qO nb8_data.tar.gz "https://ckl-emr-bucket.s3.us-east-1.amazonaws.com/vlm-workshop/nb8_data.tar.gz"
    !tar xzf nb8_data.tar.gz && rm nb8_data.tar.gz

    print("Setup complete — all dependencies and data downloaded.")


In [ ]:
import sys
from pathlib import Path

# Add json-ocr-demo directory to Python path
sys.path.insert(0, '.')

In [ ]:
import os
import json
import time
import base64
from pathlib import Path
from collections import Counter, defaultdict
from io import BytesIO
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from PIL import Image
from datasets import load_dataset
from IPython.display import HTML, display

# Import our dark-mode friendly utilities
from utils import *
from utils.config import VLLM_BASE_URL, OUTPUT_DIR

# ── Configuration ────────────────────────────────────────────────────────────
NB_NAME    = "nb8_kyc"
USE_CACHE  = True   # Set False to re-run inference and refresh cache
VLM_MODEL  = "jsl_vision_ocr_1.0"  # Best performer on MIDV-2020
set_cache_model(VLM_MODEL)

# Models with cached results (run_kyc_models.py populates these)
COMPARE_MODELS = [
    "anthropic__claude-opus-4.5",
    "anthropic__claude-sonnet-4.6",
    "openai__gpt-5.4",
    "jsl_vision_ocr_1.0",
]

print(f"✅ Configuration loaded — {VLLM_BASE_URL}")
print(f"💾 Cache: {'ON' if USE_CACHE else 'OFF'} (model={VLM_MODEL})")
print(f"📊 Comparison models: {len(COMPARE_MODELS)}")

In [ ]:
from utils.config import VLLM_MODEL as LOCAL_MODEL
if not USE_CACHE:
    check_server(VLLM_BASE_URL, LOCAL_MODEL)

---
## 🔧 Inference Engine

In [ ]:
from utils.core import infer_image_structured

print("✅ Inference engine ready — response_format json_schema (structured prediction)")

---
## 🆔 KYC Identity Verification

**The Business Problem**: Financial institutions spend `$500M+` yearly on KYC compliance. Manual verification is slow and error-prone. Fraud costs 5-10% of onboarding.

**The JSON-OCR Solution**: Instant document classification, data extraction, and quality validation.

In [ ]:
show_use_case('kyc_compliance')

In [ ]:
# KYC Schema - aligned with MIDV-2020 GT fields for accuracy validation
# Optional fields use ["string", "null"] so guided decoding allows null for absent fields.
_str   = {"type": "string"}
_str_n = {"type": ["string", "null"]}   # nullable for fields that may not appear on doc

KYC_SCHEMA = {
    "type": "object",
    "additionalProperties": False,   # force model to use only our key names
    "properties": {
        # Document metadata
        "document_type":    _str,
        "language":         _str,

        # Personal information
        "surname":          _str,
        "surname_en":       _str_n,
        "name":             _str,
        "name_en":          _str_n,
        "birth_date":       _str_n,
        "birth_place":      _str_n,
        "birth_place_en":   _str_n,
        "gender":           _str_n,
        "nationality":      _str_n,
        "nationality_en":   _str_n,

        # Document numbers
        "id_number":        _str_n,
        "number":           _str_n,

        # Dates
        "issue_date":       _str_n,
        "expiry_date":      _str_n,

        # Authority
        "authority":        _str_n,
        "authority_en":     _str_n,

        # MRZ lines (passports / some IDs)
        "mrz_line0":        _str_n,
        "mrz_line1":        _str_n,
        "mrz_line2":        _str_n,
    },
    "required": ["document_type", "surname", "name"],
}

# Fields we can validate against MIDV-2020 GT annotations
GT_FIELDS = [
    'surname', 'name', 'birth_date', 'gender', 'number', 'expiry_date',
    'issue_date', 'nationality', 'id_number', 'birth_place', 'authority',
    'mrz_line0', 'mrz_line1',
]

print(f"✅ KYC Schema: {len(KYC_SCHEMA['properties'])} fields (additionalProperties=False)")
print(f"📊 GT-validatable fields: {len(GT_FIELDS)}")

In [ ]:
from utils.kyc import load_midv_dataset, build_annotated_preview, load_photo_matches

MIDV_ROOT = Path('dataset/MIDV2020')

kyc_images, kyc_paths, kyc_metadata = load_midv_dataset(MIDV_ROOT, samples_per=1)
photo_matches = load_photo_matches('dataset/photo_scan_matches.json')
preview_images, preview_docs = build_annotated_preview(
    kyc_images, kyc_paths, kyc_metadata, MIDV_ROOT, photo_matches
)
display_document_analysis(preview_images, preview_docs)

In [ ]:
from utils.kyc import run_kyc_extraction

KYC_PROMPT = """Extract all identity fields from this document (passport or national ID card).
Read all visible text fields including names, dates, numbers, and MRZ lines.
For non-Latin scripts, also provide English transliterations in the _en fields.
Use null for fields not visible on this document."""

if USE_CACHE and has_nb_cache(NB_NAME, "kyc_results"):
    cached = load_nb_cache(NB_NAME, "kyc_results")
    kyc_results = cached["results"]
    # Restore None entries
    kyc_results = [r if r is not None and r != "null" else None for r in kyc_results]
    success = sum(1 for r in kyc_results if r is not None)
    print(f"📦 Using cached KYC results — {success}/{len(kyc_results)} successful")
else:
    kyc_results = run_kyc_extraction(
        infer_image_structured, kyc_images, kyc_paths, kyc_metadata,
        KYC_PROMPT, KYC_SCHEMA,
    )
    # Cache — convert Path objects in _meta to strings for JSON serialization
    cacheable = []
    for r in kyc_results:
        if r is None:
            cacheable.append(None)
        else:
            cr = dict(r)
            if '_meta' in cr:
                cr['_meta'] = {k: str(v) if isinstance(v, Path) else v for k, v in cr['_meta'].items()}
            cacheable.append(cr)
    save_nb_cache(NB_NAME, "kyc_results", {"results": cacheable})

In [ ]:
from utils.kyc import load_kyc_model_results, show_kyc_model_comparison

all_results = load_kyc_model_results(NB_NAME, "kyc_results", COMPARE_MODELS)
set_cache_model(VLM_MODEL)  # restore active model

show_kyc_model_comparison(
    all_results, MIDV_ROOT, GT_FIELDS,
    photo_matches=photo_matches,
)

In [ ]:
from utils.kyc import display_kyc_predictions

display_kyc_predictions(
    kyc_images, kyc_results, MIDV_ROOT,
    photo_matches=photo_matches,
    gt_fields=GT_FIELDS,
)

---
## Impact & ROI

### Model Comparison (MIDV-2020 — 20 ID Documents, 10 Countries, 10 Scripts)

| Model | Overall | Scans | Photos | Extraction |
|-------|:---:|:---:|:---:|:---:|
| **Claude Opus 4.5** | **68.3%** | **91.9%** | **45.0%** | 20/20 |
| **JSL Vision OCR 1.0** | **62.3%** | **89.9%** | 35.0% | 20/20 |
| Claude Sonnet 4.6 | 60.3% | 83.8% | 31.2% | 18/20 |
| GPT-5.4 | 53.8% | 76.8% | 31.0% | 20/20 |

### JSL Vision OCR 1.0 — Per-Field Accuracy

| Field | Accuracy | Notes |
|-------|:---:|-------|
| Expiry date | **87.5%** | Best field — dates are structured |
| MRZ line 1 | **77.8%** | Machine-readable zone |
| Birth date | **75.0%** | Consistently formatted |
| Gender | **75.0%** | Short field, easy to read |
| ID number | **71.4%** | Alphanumeric, well-separated |
| Surname | **70.0%** | Good on Latin scripts |
| Name | 61.1% | Script diversity is the bottleneck |
| Issue date | 61.1% | Sometimes confused with other dates |
| Number | 60.0% | Varies by country |
| Birth place | 45.5% | Free-text, multilingual |
| Nationality | 42.9% | Varies widely across countries |
| MRZ line 0 | 35.7% | Long strings, small print |
| Authority | 33.3% | Often abbreviated, multilingual |

**Honest assessment**: JSL Vision OCR 1.0 matches Claude Sonnet 4.6 and beats GPT-5.4 on this deliberately hard 10-country multilingual benchmark. On clean scans all top models exceed 80% — camera photos (angle, blur, lighting) are the universal bottleneck. Production deployments targeting 1–2 languages will significantly outperform these numbers.

### Speed

| Metric | Manual Entry | VLM Extraction |
|--------|:-----------:|:--------------:|
| Time per document | 2–3 min | **< 3 sec** |
| Batch of 100 IDs | 3–5 hours | **< 5 min** |
| Languages supported | Staff-dependent | **Any** (zero-shot) |

### Cost Savings

*Assumptions: KYC compliance analyst at &#36;35/hr, 250 working days/yr. Industry reference: financial institutions spend &#36;500M+/yr on KYC compliance (Thomson Reuters).*

| Scale | Manual Cost | Automated Cost | Annual Savings |
|-------|:----------:|:--------------:|:--------------:|
| 500 IDs/month | &#36;14.6K/yr | ~&#36;600/yr | **~&#36;14K/yr** |
| 10,000 IDs/month | &#36;175K/yr | ~&#36;3K/yr | **~&#36;172K/yr** |
| 100,000 IDs/month (large bank) | &#36;1.75M/yr | ~&#36;15K/yr | **~&#36;1.7M/yr** |

### Why This Matters

- **KYC compliance**: automated field extraction + validation for financial onboarding
- **Competitive with frontier models**: JSL Vision OCR 1.0 matches Claude Sonnet 4.6 and outperforms GPT-5.4 — fully on-prem, no cloud API needed
- **Multi-country**: 10 countries demonstrated, extensible to any ID format without retraining
- **Translation**: automatic English transliteration of non-Latin fields (20/20 docs)
- **Audit trail**: every extraction produces structured, timestamped JSON for regulatory review
